# 프롬프트 엔지니어링을 위한 모범 사례

출처: https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-openai-api


In [1]:
import re
import requests
import sys
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
import tiktoken
from dotenv import load_dotenv
load_dotenv()

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

client = AzureOpenAI(
  azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT"), 
  azure_ad_token_provider=token_provider,
  api_version="2024-02-15-preview"
)

CHAT_COMPLETIONS_MODEL = os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')

# 1. 최신 모델 사용

최고의 결과를 위해 최신 모델을 사용하세요.

# 2. 프롬프트의 시작 부분에 지침을 배치하고 ### 또는 """로 지침과 컨텍스트를 구분하세요

In [2]:
response = client.chat.completions.create(
        model=CHAT_COMPLETIONS_MODEL,
        messages = [{"role":"system", "content":"당신은 도움이 되는 어시스턴트입니다."},
                                {"role":"user","content": '아래 텍스트를 가장 중요한 요점의 글머리표 목록으로 한글로 요약해 주세요. \n\n \
                ###\n\nWe’re happy to announce that OpenAI and Microsoft are extending our partnership.\
                This multi-year, multi-billion dollar investment from Microsoft follows their previous investments \
                in 2019 and 2021, and will allow us to continue our independent research and develop AI that is \
                increasingly safe, useful, and powerful. \n\n \
                In pursuit of our mission to ensure advanced AI benefits all of humanity, OpenAI remains a \
                capped-profit company and is governed by the OpenAI non-profit. This structure allows us to \
                raise the capital we need to fulfill our mission without sacrificing our core beliefs about \
                broadly sharing benefits and the need to prioritize safety. \
                Microsoft shares this vision and our values, and our partnership is instrumental to our progress. \n###',}],
                max_tokens=400,)

print(response.choices[0].message.content)

- OpenAI와 Microsoft가 파트너십을 연장하기로 발표함
- Microsoft가 수년간 수십억 달러를 추가 투자하며, 2019년과 2021년 투자에 이어짐
- 이 투자로 OpenAI가 독립적인 연구와 더욱 안전하고 유용하며 강력한 AI 개발을 계속할 수 있게 됨
- OpenAI는 모든 인류에게 고도 AI의 혜택을 제공하는 것이 목표이며, 제한적 이익 구조와 비영리 조직이 운영함
- 이러한 구조를 통해 핵심 가치(이익 공유, 안전 우선)를 유지하면서 필요한 자본을 조달할 수 있음
- Microsoft는 OpenAI의 사명과 가치에 공감하며 파트너십이 발전에 중요한 역할을 함


# 3. 원하는 컨텍스트, 결과, 길이, 형식, 스타일 등에 대해 구체적이고 상세하게 작성하세요

In [3]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": '경봉궁을 위한 시 한편 써줘',}],
        max_tokens=400,)

print(response.choices[0].message.content)


경봉궁을 위한 시

밤하늘 아래 경봉궁,  
달빛이 물든 담장 위로  
고요한 시간이 흐른다.  
유구한 역사의 숨결 속에  
긴 세월의 바람도 스쳐간다.

청아한 소리의 바람결,  
산뜻한 푸름이 머무는 궁.  
벚꽃이 흩날리는 봄날엔  
왕의 마음도 흔들렸으리라.

경봉궁이여, 너의 아름다움  
찬란한 지난날의 꿈을 안고  
오늘을 살아가는 우리 곁에  
늘 변함없이 서 있구나.

세월이 흘러도  
경봉궁의 위엄과 단아함은  
마음 깊은 곳에 잔잔히 스며  
우리들의 역사가 된다.


In [4]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": '경봉궁을 위한 시 한편 써줘 \
                특히 봄에서 여름으로 가는 게절에 아름다운 풍경을 충분히 묘사해줘',}],
        max_tokens=400,)

print(response.choices[0].message.content)



경봉궁의 봄과 여름 사이  
한가로운 바람에 휘날리는  
대청마루의 얇은 창살 사이로  
노란 햇빛이 스며든다

진달래 수줍게 고개 들고  
벚꽃은 마치 눈처럼 쏟아진 뒤  
초록의 물결이 궁 담을 넘는다  
계단 아래 작은 연못에는  
개구리 울음이 가득 번지고

산새는 푸른 하늘을 헤엄치고  
궁전의 처마 끝에 매달린 온기  
매일 새로이 피는 풀꽃이  
봄의 끝에 여름을 부른다

경봉궁의 하루는 빛으로 가득 차  
흔들리는 나뭇잎, 그늘 아래  
고요한 시간을 따라 걷다 보면  
계절은 어느덧 여름의 문 앞에  
한 송이 노란 해바라기를 펼친다


# 4. 원하는 출력 형식을 예제를 통해 명확히 표현하세요 (예제 1, 예제 2).

In [5]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": '아래 텍스트에서 회사명 다음 연도를 추출하고 각 엔티티의 시작 인덱스와 끝 인덱스를 출력합니다.\
                Generate output as {"text": "OpenAI", "start": 28, "end": 34} \
                ###\
                OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 전하게 되어 기쁩니다. \
                Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년에 이루어진 이전 투자에 이은 것입니다. \
                2019년과 2021년 투자에 이은 것으로, 이를 통해 우리는 독립적인 연구를 계속하고 더욱 안전하고 유용하며 강력한 \
                더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다. \n\n \
                ###\
                ',}],
                
        max_tokens=400,)

print(response.choices[0].message.content)



[{"text": "Microsoft", "start": 53, "end": 62, "year": "2019"}, {"text": "Microsoft", "start": 53, "end": 62, "year": "2021"}]


In [6]:
prompt = """
                아래 텍스트에 언급된 엔티티를 추출합니다. 
                아래 텍스트에 언급된 중요한 엔티티를 추출합니다. 
                먼저 모든 회사 이름을 추출한 다음 모든 연도를 추출합니다, 
                그런 다음 콘텐츠에 맞는 특정 주제를 추출하고 마지막으로 일반적인 주요 주제를 추출합니다.\n\n 
                Desired format: 
                Company names: <comma_separated_list_of_company_names> 
                Years: -||- 
                Specific topics: -||- 
                General themes: -||- 

                ###\n\n
                OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 알려드리게 되어 기쁩니다.
                Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년 투자에 이어 
                2019년과 2021년 투자에 이은 것으로, 이를 통해 우리는 독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 
                더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다. \n\n

        """

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": prompt}],
        max_tokens=400,)

print(response.choices[0].message.content)



Company names: OpenAI, Microsoft  
Years: 2019, 2021  
Specific topics: partnership extension, multi-year multibillion-dollar investment, development of safe and powerful AI, independent research  
General themes: artificial intelligence, corporate partnership, technology investment


# 5. 제로샷(zero-shot)으로 시작하고, 이후 몇 가지 예제를 제공하세요. 둘 다 효과가 없으면 미세 조정을 시도하세요 (업데이트 필요)

In [7]:
prompt = """
        OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 알려드리게 되어 기쁩니다.
        Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년 투자에 이어 
        2019년과 2021년 투자에 이은 것으로, 이를 통해 우리는 독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 
        더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다.

"""

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant. Extract keywords from the corresponding texts below."},
                {"role":"user","content": prompt,}],
        max_tokens=400,)

print(response.choices[0].message.content)



OpenAI, Microsoft, 파트너십 연장, 투자, 다년간, 수십억 달러, 독립적 연구, 안전한 AI, 유용한 AI, 강력한 AI, 2019년, 2021년


In [8]:
system_prompt = """

        당신은 유용한 조수입니다. 아래 해당 텍스트에서 키워드를 추출하세요..
        텍스트: 스트라이프는 웹 개발자가 웹 사이트와 모바일 애플리케이션에 결제 처리를 통합하는 데 사용할 수 있는 API를 제공합니다. 
        키워드: 스트라이프, 결제 처리, API, 웹 개발자, 웹사이트, 모바일 애플리케이션 
        ###
        텍스트: OpenAI는 텍스트를 이해하고 텍스트를 생성하는 데 매우 능숙한 최첨단 언어 모델을 학습시켰습니다.
        이해하고 텍스트를 생성하는 데 매우 능숙한 최첨단 언어 모델을 학습시켰습니다. API를 통해 이러한 모델에 액세스할 수 있으며, 언어 처리와 관련된 거의 모든 작업을
        언어 처리와 관련된 모든 작업을 해결하는 데 사용할 수 있습니다.
        키워드: 언어 모델, 텍스트 처리, API.
        
"""

user_prompt = """
        텍스트: OpenAI와 Microsoft가 파트너십을 연장한다는 소식을 전하게 되어 기쁘게 생각합니다.
        Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년에 이루어진 이전 투자에 이은 것입니다.
        2019년과 2021년 투자에 이은 것으로, 이를 통해 우리는 독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 
        더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다. 
        키워드:
"""

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content": system_prompt},
                {"role":"user","content": user_prompt,}],
        max_tokens=400,)

print(response.choices[0].message.content)



키워드: OpenAI, Microsoft, 파트너십, 투자, AI 개발, 독립적 연구, 안전한 AI, 유용한 AI, 강력한 AI


# 6. 모호하고 부정확한 설명을 줄이세요

In [10]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": '새 제품에 대한 설명을 작성하세요. 이 제품은 차세대 카시트입니다. 이 제품에 대한 설명은 몇 문장으로만 짧게 작성하고 너무 길지 않아야 합니다.',}],
        max_tokens=400,)

print(response.choices[0].message.content)



이 차세대 카시트는 최첨단 안전 기술과 편리한 설치 시스템을 갖추고 있습니다. 아이의 성장에 따라 조절이 가능하며, 통기성 소재로 쾌적함을 제공합니다. 스타일리시한 디자인으로 차량 인테리어와도 잘 어울립니다.


In [11]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": '새 제품에 대한 설명을 작성하세요. 이 제품은 차세대 카시트입니다. 3~5문장으로 구성된 단락을 사용하여 이 제품을 설명하세요.',}],
        max_tokens=400,)

print(response.choices[0].message.content)



이 차세대 카시트는 어린이의 안전과 편안함을 극대화하기 위해 설계되었습니다. 인체공학적으로 디자인된 구조와 고급 소재를 사용하여 장시간 사용에도 피로감을 최소화합니다. 스마트 센서가 아이의 착석 상태를 실시간으로 감지하고, 위험 상황에서는 알림을 제공합니다. 설치가 간편하며, 다양한 차량 모델에 호환 가능한 점도 큰 장점입니다. 혁신적인 기능과 현대적인 디자인을 결합한 이 제품은 부모와 아이 모두에게 새로운 만족을 선사합니다.


# 7. 하지 말아야 할 것을 말하는 대신, 해야 할 것을 명확히 설명하세요

In [12]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content": 'The following is a conversation between an Agent and a Customer. DO NOT ASK USERNAME OR PASSWORD. DO NOT REPEAT. Customer: I can’t log in to my account.\
                Agent:',}],
        max_tokens=400,)

print(response.choices[0].message.content)


I'm sorry you're having trouble logging in. Could you let me know if you're seeing any error messages or if you've tried resetting your password already? I'll do my best to help you get back into your account.


In [13]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content":'The following is a conversation between an Agent and a Customer. \
                The agent will attempt to diagnose the problem and suggest a solution, whilst refraining from asking any questions related to PII. \
                Instead of asking for PII, such as username or password, refer the user to the help article www.samplewebsite.com/help/faq \n\n\
                Customer: I can’t log in to my account. \n\
                Agent:',}],
        max_tokens=400,)

print(response.choices[0].message.content)


I'm sorry you're having trouble logging in to your account. If you're experiencing issues such as forgotten credentials or account access errors, I recommend visiting our help article at www.samplewebsite.com/help/faq for guidance. You'll find step-by-step instructions to resolve most login problems there. If the issue persists after following the help article, please let me know and I'll be happy to assist further.


# 8. 코드 생성 관련 - 특정 패턴으로 모델을 유도하기 위해 "선도 단어"를 사용하세요

In [14]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages = [{"role":"system", "content":"You are a helpful assistant."},
                {"role":"user","content":'# 다음과 같은 간단한 파이썬 함수를 작성합니다. \n\
                # 1. 마일 단위로 숫자를 요청하세요.\n\
                # 2. 마일을 킬로미터로 변환합니다.',}],
        max_tokens=400,)

print(response.choices[0].message.content)


다음은 요청하신 간단한 파이썬 함수 예제입니다:

def miles_to_kilometers():
    miles = float(input("마일 단위의 숫자를 입력하세요: "))
    kilometers = miles * 1.60934
    print(f"{miles} 마일은 {kilometers:.2f} 킬로미터입니다.")

함수 사용 방법  
함수를 호출하면 숫자를 입력하라는 메시지가 나타나고, 마일 값을 입력하면 킬로미터로 변환해 출력합니다.

예시  
마일 단위의 숫자를 입력하세요: 10  
10 마일은 16.09 킬로미터입니다.

설명이 필요하시거나 추가 기능을 원하시면 말씀해 주세요!
